In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import time

In [27]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_7424\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


In [28]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Imputing missing values ##

In [29]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [30]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [31]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [32]:
df.drop(['Store','Date','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [33]:
df.sample(10)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
129233,2,5109,475,1,0,0,1,a,a,230.0,1,2015,4,7,0,129.0,15,13.25,1
747792,5,7413,887,1,1,0,1,a,c,760.0,0,2013,8,30,0,0.0,35,0.00,0
352506,5,9596,1139,1,0,0,0,d,c,6680.0,0,2014,8,29,0,11.0,35,0.00,0
822988,1,6316,572,1,0,0,0,d,c,570.0,1,2013,6,24,0,0.0,26,27.00,0
405214,5,5563,550,1,1,0,0,a,c,610.0,1,2014,7,4,0,127.0,27,9.75,0
234901,5,4489,420,1,0,0,1,a,a,970.0,1,2015,1,2,0,22.0,1,16.50,0
214898,2,4518,463,1,0,0,0,a,c,720.0,0,2015,1,20,0,3.0,4,0.00,0
18149,3,10752,834,1,1,0,0,a,c,2290.0,1,2015,7,15,1,30.0,29,16.75,0
224128,7,0,0,0,0,0,0,a,a,1300.0,1,2015,1,11,0,10.0,2,38.50,1
716000,6,2410,326,1,0,0,0,a,a,2330.0,1,2013,9,28,0,0.0,39,29.25,0


In [35]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [36]:
df.sample(20)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
551563,6,6020,699,1,0,0,0,d,c,1250.0,1,2014,2,22,1,13.0,8,28.00,0
891813,2,9937,888,1,1,0,0,d,c,580.0,0,2013,4,23,0,0.0,17,0.00,0
910915,6,4831,466,1,0,0,0,d,c,4330.0,1,2013,4,6,0,26.0,14,18.75,0
968542,3,3810,444,1,0,0,0,d,c,4820.0,0,2013,2,13,0,59.0,7,0.00,0
969718,2,5573,777,1,0,0,0,a,a,26450.0,0,2013,2,12,1,1.0,7,0.00,0
223503,1,8488,1135,1,1,0,0,c,c,820.0,0,2015,1,12,1,24.0,3,0.00,0
5873,7,4865,517,1,0,0,0,d,c,38630.0,0,2015,7,26,0,34.0,30,0.00,0
597076,7,0,0,0,0,0,0,a,a,2170.0,0,2014,1,12,0,62.0,2,0.00,0
564910,1,5027,691,1,0,0,0,a,a,460.0,1,2014,2,10,0,0.0,7,6.00,1
518886,7,0,0,0,0,0,0,a,a,22390.0,1,2014,3,23,0,71.0,12,53.75,0


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 19 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   DayOfWeek                1017209 non-null  int64  
 1   Sales                    1017209 non-null  int64  
 2   Customers                1017209 non-null  int64  
 3   Open                     1017209 non-null  int64  
 4   Promo                    1017209 non-null  int64  
 5   StateHoliday             1017209 non-null  int64  
 6   SchoolHoliday            1017209 non-null  int64  
 7   StoreType                1017209 non-null  object 
 8   Assortment               1017209 non-null  object 
 9   CompetitionDistance      1017209 non-null  float64
 10  Promo2                   1017209 non-null  int64  
 11  Year                     1017209 non-null  int32  
 12  Month                    1017209 non-null  int32  
 13  Day                      1017209 non-null 

In [39]:
num_col=['Customers','CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths']
cat_col = ['StoreType','Assortment','Year']

In [40]:
X = df.drop(columns=['Sales'])
y = df['Sales']

In [41]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [42]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [50]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [ ]:
models = {
    "XGBRegressor" : XGBRegressor(),
    'RandomForestRegressor': RandomForestRegressor(),
    'LinearRegression': LinearRegression()
}

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)
    model.fit(X_train,y_train)
    start = time.perf_counter()
    prediction = model.predict(X_test)
    stop = time.perf_counter()
    time_taken = stop-start
    rmse = root_mean_squared_error(y_test,prediction)
    rmsep = rmse/y_test_mean

    print(f'{model_name}: {rmsep*100:.2f}%\n')
    print(f'Time taken for {model_name}: {time_taken}\n')

XGBRegressor: 13.47%

Time taken for XGBRegressor: 0.16958928108215332

RandomForestRegressor: 10.31%

Time taken for RandomForestRegressor: 62.137805223464966

LinearRegression: 23.75%

Time taken for LinearRegression: 0.14775538444519043

